# Partie 2 : Classification de texte avec Transformer
Master 1 IdL - 2025/2026

On utilise le modele **CamemBERT** pour predire la categorie d'un article de presse.

In [ ]:
# Installation (a executer une seule fois)
!pip install transformers torch datasets pandas scikit-learn

In [ ]:
# Importations
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from transformers import AutoTokenizer, AutoModelForSequenceClassification

print("OK")

## Etape 1 : Chargement du dataset

### Structure du dataset AVANT tokenisation

Le fichier CSV contient 3 colonnes :

| Colonne | Type | Description | Exemple |
|---|---|---|---|
| url | str | adresse de l'article | https://www.francetvinfo.fr/... |
| texte | str | contenu de l'article | "Le gouvernement a annonce..." |
| label | str | categorie de l'article | politique, sport, economie... |

Apres encodage des labels :

| label (str) | label_id (int) |
|---|---|
| economie | 0 |
| politique | 1 |
| sante | 2 |
| sciences | 3 |
| sport | 4 |

In [ ]:
# Chargement du dataset
df = pd.read_csv("articles.csv")

print("Nombre d'articles : " + str(len(df)))
print("Colonnes : " + str(list(df.columns)))
print("")
print("Repartition par label :")
print(df["label"].value_counts())
print("")
print("Longueur moyenne des textes : " + str(int(df["texte"].str.len().mean())) + " caracteres")
print("Longueur min : " + str(df["texte"].str.len().min()) + " caracteres")
print("Longueur max : " + str(df["texte"].str.len().max()) + " caracteres")

# Conversion labels texte -> nombres
le = LabelEncoder()
df["label_id"] = le.fit_transform(df["label"])

print("")
print("Apercu du dataset :")
df.head(3)

## Etape 2 : Division train / test

In [ ]:
# Division 80% train / 20% test
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

print("Train : " + str(len(train_df)) + " articles")
print("Test  : " + str(len(test_df)) + " articles")

## Etape 3 : Tokenisation

### Structure du dataset APRES tokenisation

Le tokeniseur convertit chaque texte en une sequence de nombres (tokens).

| Champ | Type | Description |
|---|---|---|
| input_ids | list[int] | texte converti en tokens (nombres) |
| attention_mask | list[int] | 1 = vrai token, 0 = padding |

Exemple concret :
```
Texte          : "Le PSG a gagne"
input_ids      : [5, 1234, 456, 78, 910, 6]
attention_mask : [1,    1,   1,  1,   1, 1]
```

Si le texte est trop court (padding) :
```
Texte          : "OK"
input_ids      : [5, 99, 6, 0, 0, 0, 0 ...]
attention_mask : [1,  1, 1, 0, 0, 0, 0 ...]
```

In [ ]:
# Chargement du tokeniseur CamemBERT
tokenizer = AutoTokenizer.from_pretrained("camembert-base")

# Exemple de tokenisation sur un vrai texte
exemple = "Le gouvernement a annonce une reforme."
tokens = tokenizer(exemple, padding="max_length", truncation=True, max_length=20)

print("Texte original :")
print(exemple)
print("")
print("input_ids (texte converti en nombres) :")
print(tokens["input_ids"])
print("")
print("attention_mask (1=vrai token, 0=padding) :")
print(tokens["attention_mask"])

## Etape 4 : Chargement du modele et prediction

### Sorties du modele : les logits

Le modele produit des **logits** — ce sont des scores bruts pour chaque categorie.

```
Exemple avec 5 categories :

logits = [-1.2,  0.3,  4.5, -0.8,  0.1]
           |     |     |     |     |
        economie sport sante sciences politique

Le score le plus eleve est 4.5 → prediction = sante
```

On utilise **argmax** pour trouver l'indice du score le plus eleve.

In [ ]:
# Chargement du modele
nb_labels = len(le.classes_)
modele = AutoModelForSequenceClassification.from_pretrained(
    "camembert-base",
    num_labels=nb_labels
)

def predire(texte):
    """
    Predit la categorie d'un texte.
    Entree : texte (str)
    Sortie : label predit (str), logits (list[float])
    """
    # Tokenisation du texte
    tokens = tokenizer(
        texte,
        return_tensors="pt",
        truncation=True,
        max_length=128
    )

    # Prediction du modele
    with torch.no_grad():
        sorties = modele(**tokens)

    # Les logits sont les scores bruts pour chaque categorie
    logits = sorties.logits

    # On prend la categorie avec le score le plus eleve
    prediction = torch.argmax(logits, dim=1).item()
    label_predit = le.classes_[prediction]

    return label_predit, logits.numpy().tolist()[0]

print("Modele charge")

In [ ]:
# Test sur des exemples
textes = [
    "Le gouvernement annonce une reforme des retraites.",
    "Le PSG a gagne le match 3 a 1.",
    "Un nouveau vaccin est disponible contre la grippe.",
    "La croissance economique ralentit en Europe.",
    "Des chercheurs ont decouvert une nouvelle planete.",
]

print("--- TESTS DE PREDICTION ---")
print("")

for texte in textes:
    label, logits = predire(texte)
    print("Texte   : " + texte)
    print("Logits  : " + str([round(x, 2) for x in logits]))
    print("Labels  : " + str(list(le.classes_)))
    print("Predit  : " + label)
    print("")

## Recap : Toutes les structures de donnees

### 1. Dataset brut (CSV)
```
DataFrame pandas :
  texte    (str) : texte brut de l'article
  label    (str) : categorie (politique, sport...)
  label_id (int) : categorie en nombre (0, 1, 2...)
```

### 2. Dataset apres tokenisation
```
Dict Python :
  input_ids      (list[int]) : texte converti en tokens
  attention_mask (list[int]) : 1=vrai token, 0=padding
```

### 3. Sorties du modele
```
logits (tensor) : scores bruts pour chaque categorie
  ex : [-1.2, 0.3, 4.5, -0.8, 0.1]
  → argmax → indice 2 → label = sante
```

### Complexite
```
Tokenisation : O(L) ou L = longueur du texte
Prediction   : O(L x H) ou H = taille cachee du modele (768 pour CamemBERT)
```